# 02 - EDA: COVID-QU-Ex

Misma estructura que los notebooks anteriores (ver `00_eda_chest_xray.ipynb`). Lo que cambia aquí: las etiquetas vienen de carpetas por clase (como Chest_x_ray), pero la fuente trae **dos conjuntos distintos** (`Lung Segmentation Data` y `Infection Segmentation Data`) y además máscaras de segmentación pulmonar — hay que entender bien la relación entre esos dos conjuntos antes de contar nada, porque uno es subconjunto del otro.

## 0. Configuración

In [ ]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm import tqdm

sys.path.insert(0, ".")
import eda_utils as eu

tqdm.pandas()

DATASET_NAME = "COVID-QU-Ex"
DATASET_DIR = Path("../Datasets/COVID-QU-Ex-Dataset")
LUNG_SEG_DIR = DATASET_DIR / "Lung Segmentation Data" / "Lung Segmentation Data"
INFECTION_SEG_DIR = DATASET_DIR / "Infection Segmentation Data" / "Infection Segmentation Data"
SPLITS = ["Train", "Val", "Test"]
SOURCE_CLASSES = ["COVID-19", "Non-COVID", "Normal"]

# Carpeta de origen -> clase objetivo del proyecto.
# "Non-COVID" NO es Normal: según el propio dataset (léase el .txt del dataset) son "Non-COVID
# infections (Viral or Bacterial Pneumonia)" -> es Pneumonia, no una cuarta categoría.
CLASS_MAP = {
    "COVID-19": "COVID-19",
    "Normal": "Normal",
    "Non-COVID": "Pneumonia",
}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("../resultados/eda") / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HASH_DIR = Path("../Datasets/_eda_hashes")
HASH_DIR.mkdir(parents=True, exist_ok=True)

## 1. Inventario básico
Se inventaría `Lung Segmentation Data` (el conjunto completo, con máscara pulmonar para el 100% de las imágenes) y por separado `Infection Segmentation Data`, para poder compararlos en la siguiente sección.

In [ ]:
def inventory(base_dir: Path, mask_folders):
    records = []
    for split in SPLITS:
        for cls in SOURCE_CLASSES:
            img_dir = base_dir / split / cls / "images"
            if not img_dir.exists():
                continue
            for f in eu.list_images(img_dir):
                rec = {"split": split, "source_class": cls, "fname": f.name, "path": str(f)}
                for mask_name in mask_folders:
                    mask_path = base_dir / split / cls / mask_name / f.name
                    rec[f"has_{mask_name.replace(' ', '_')}"] = mask_path.exists()
                records.append(rec)
    return pd.DataFrame(records)

lung_seg = inventory(LUNG_SEG_DIR, ["lung masks"])
infection_seg = inventory(INFECTION_SEG_DIR, ["lung masks", "infection masks"])

print(f"Lung Segmentation Data: {len(lung_seg)} imágenes (esperado por el .txt del dataset: 33,920)")
print(f"Infection Segmentation Data: {len(infection_seg)} imágenes (esperado: 5,826)")

In [ ]:
print("Lung Segmentation Data por split/clase:")
display(lung_seg.groupby(["split", "source_class"]).size().unstack(fill_value=0))
print()
print("¿Todas las imágenes de Lung Segmentation Data tienen su máscara pulmonar?")
print(lung_seg["has_lung_masks"].value_counts())

### 1.1 ¿`Infection Segmentation Data` son imágenes nuevas, o el mismo subconjunto de arriba?

El `.txt` del propio dataset dice que es "a subset of COVID-QU-Ex dataset" — se verifica aquí en vez de asumirlo: se cruzan los nombres de archivo de `Infection Segmentation Data` contra los de `Lung Segmentation Data`, por split y clase.

In [ ]:
overlap_rows = []
for split in SPLITS:
    for cls in SOURCE_CLASSES:
        inf_names = set(infection_seg[(infection_seg.split == split) & (infection_seg.source_class == cls)]["fname"])
        lung_names = set(lung_seg[(lung_seg.split == split) & (lung_seg.source_class == cls)]["fname"])
        overlap_rows.append({
            "split": split,
            "source_class": cls,
            "n_infection_seg": len(inf_names),
            "n_dentro_de_lung_seg": len(inf_names & lung_names),
            "es_subconjunto_completo": inf_names.issubset(lung_names),
        })
overlap_df = pd.DataFrame(overlap_rows)
overlap_df

**Confirmado**: `Infection Segmentation Data` es 100% subconjunto de `Lung Segmentation Data` (mismos nombres de archivo). Conclusión práctica: **no se trata como una fuente aparte** — se cuenta solo `Lung Segmentation Data` como pool de imágenes utilizables, y se deja registrado qué subconjunto de esas imágenes tiene además máscara de infección (útil más adelante para explicabilidad, no para el conteo de clases).

In [ ]:
df = lung_seg.copy()
infection_keys = set(zip(infection_seg["split"], infection_seg["source_class"], infection_seg["fname"]))
df["has_infection_mask"] = [
    (row.split, row.source_class, row.fname) in infection_keys for row in df.itertuples()
]
print(f"Imágenes con máscara de infección disponible: {df['has_infection_mask'].sum()} de {len(df)}")

## 2. Esquema de etiquetas y mapeo a clases objetivo

**Hallazgo importante**: en el notebook de fusión (`01_dataset_preprocessing.ipynb`) esta fuente solo se usaba para COVID-19 y Normal — la clase `Non-COVID` (11,263 imágenes) se ignoraba por completo. Pero según el `.txt` del propio dataset, `Non-COVID` = *"Non-COVID infections (Viral or Bacterial Pneumonia)"*, es decir, **es Pneumonia**, no una categoría aparte. Eso significa que ya había más de 11 mil imágenes de Pneumonia de población adulta disponibles en un dataset que ya habían descargado, justo lo que se estaba buscando en fuentes externas (NIH ChestX-ray14, etc.) para diversificar Pneumonia más allá del origen pediátrico de Kermany.

**Con una salvedad que hay que verificar, no asumir**: los datasets de COVID-19 de Qatar University (de donde viene COVID-QU-Ex) históricamente han reconstruido su clase de neumonía viral/bacteriana reutilizando imágenes de fuentes ya conocidas — es plausible que parte de este `Non-COVID` se solape con el propio Chest_x_ray (Kermany) o con RSNA_Pneumonia, ambos ya en este proyecto. Esto **no se puede confirmar por el nombre del dataset**, solo con el cruce de hashes MD5 que se prepara en la Sección 6 y se resolverá en el notebook de consolidación entre todas las fuentes.

In [ ]:
df["target_class"] = df["source_class"].map(CLASS_MAP)
unmapped = df[df["target_class"].isna()]
if len(unmapped):
    print("Carpetas sin mapear (revisar CLASS_MAP):", sorted(unmapped["source_class"].unique()))
else:
    print("Todas las carpetas de origen quedaron mapeadas correctamente.")

## 3. Conteo real por clase objetivo (por split)

In [ ]:
summary = df.groupby(["split", "target_class"]).size().unstack(fill_value=0)
summary = summary.reindex(columns=sorted(summary.columns))
summary.loc["Total"] = summary.sum()
summary["Total"] = summary.sum(axis=1)
summary.to_csv(OUTPUT_DIR / "conteo_por_clase.csv")
summary

In [ ]:
totales = df["target_class"].value_counts().sort_index()
ax = totales.plot(kind="bar", figsize=(6, 4), color="#C44E52")
ax.set_title(f"{DATASET_NAME}: total de imágenes por clase (train+val+test)")
ax.set_ylabel("Número de imágenes")
for i, v in enumerate(totales):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conteo_por_clase.png", dpi=120)
plt.show()

## 4. Calidad e integridad

In [ ]:
probe_results = [eu.probe_image(p) for p in tqdm(df["path"], desc="Verificando imágenes")]
df["width"], df["height"], df["mode"], df["error"] = zip(*probe_results)

corrupt = df[df["error"].notna()]
print(f"Imágenes corruptas / no abribles: {len(corrupt)}")
corrupt[["path", "error"]]

In [ ]:
print("Modo de color:")
display(df["mode"].value_counts())
print()
print("Dimensiones (ancho x alto):")
df[["width", "height"]].describe()

### 4.1 Candidatas a excluir por calidad de imagen (mismo criterio que en las fuentes anteriores)

In [ ]:
calidad = eu.flag_quality_issues(df[df["error"].isna()], desc=DATASET_NAME)
sospechosas = calidad[calidad["flag_bordes"] | calidad["flag_brillo"]].copy()
print(f"Candidatas marcadas para revisión manual: {len(sospechosas)} de {len(calidad)} "
      f"({100 * len(sospechosas) / len(calidad):.1f}%)")
sospechosas["target_class"].value_counts()

In [ ]:
sospechosas[["path", "target_class", "split", "content_ratio", "glare_fraction", "motivo"]].to_csv(
    OUTPUT_DIR / "candidatas_baja_calidad.csv", index=False
)
eu.show_flagged(sospechosas, save_path=OUTPUT_DIR / "candidatas_baja_calidad_muestra.png")

## 5. Muestra visual por clase

In [ ]:
eu.show_samples(df[df["error"].isna()], save_path=OUTPUT_DIR / "muestra_por_clase.png")

### 5.1 Bonus: imagen + su máscara pulmonar
Esta es la primera fuente con máscaras de segmentación pulmonar para el 100% de las imágenes — vale la pena ver un ejemplo, porque estas máscaras podrían usarse más adelante para recortar la región pulmonar (ROI) antes de entrenar, o para explicabilidad (comparar el Grad-CAM del modelo contra la máscara real).

In [ ]:
def show_image_and_mask(row, mask_folder="lung masks"):
    mask_path = LUNG_SEG_DIR / row["split"] / row["source_class"] / mask_folder / row["fname"]
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    with Image.open(row["path"]) as img:
        axes[0].imshow(img, cmap="gray")
    axes[0].set_title("Imagen")
    axes[0].axis("off")
    with Image.open(mask_path) as mask:
        axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Máscara pulmonar")
    axes[1].axis("off")
    plt.suptitle(f"{row['target_class']} - {row['fname']}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ejemplo_imagen_y_mascara.png", dpi=120)
    plt.show()

ejemplo = df[df["target_class"] == "COVID-19"].sample(1, random_state=RANDOM_SEED).iloc[0]
show_image_and_mask(ejemplo)

## 6. Riesgo de solapamiento / duplicados
Se hashea solo `Lung Segmentation Data` (ya se confirmó en la Sección 1.1 que `Infection Segmentation Data` no aporta imágenes nuevas, así que hashearla también sería trabajo redundante).

In [ ]:
df["md5"] = df["path"].progress_apply(eu.md5_file)
df["phash"] = [eu.phash_file(p) for p in tqdm(df["path"], desc="Hash perceptual")]
n_dupes = df["md5"].duplicated().sum()
print(f"Duplicados exactos dentro de este dataset: {n_dupes}")

splits_por_hash = df.groupby("md5")["split"].nunique()
leak_hashes = splits_por_hash[splits_por_hash > 1].index
print(f"Hashes que aparecen en más de un split (posible fuga train/val/test): {len(leak_hashes)}")

In [ ]:
df[["path", "md5", "phash", "target_class", "split"]].assign(source_dataset=DATASET_NAME).to_csv(
    HASH_DIR / f"{DATASET_NAME}_md5.csv", index=False
)

## 7. Metadata demográfica / poblacional

Sin edad ni sexo de paciente, igual que las fuentes anteriores. Lo que sí aporta esta fuente y ninguna otra hasta ahora:
- **Máscara de segmentación pulmonar para el 100% de las imágenes** (33,920/33,920).
- **Máscara de segmentación de la infección** para un subconjunto (5,826 imágenes, ver Sección 1.1) — potencialmente útil para explicabilidad en COVID-19 específicamente.

Procedencia (según el `.txt` del dataset): compilado por Qatar University a partir de varias publicaciones propias del mismo grupo (Tahir et al. 2021, Rahman et al. 2021, Chowdhury et al. 2020, entre otras) — es en sí mismo un dataset ya agregado de múltiples fuentes, no una única adquisición. Esto refuerza la necesidad del cruce de hashes de la Sección 6 antes de dar por definitivo cuánto Pneumonia "nuevo" aporta realmente frente a lo que ya se tiene de Chest_x_ray y RSNA_Pneumonia.

## 8. Veredicto

- **Hallazgo principal**: `Non-COVID` (11,263 imágenes) es Pneumonia de población adulta, no una clase descartable — el notebook de fusión la estaba ignorando por completo. Esto podría resolver de un solo golpe la necesidad de diversificar el origen de Pneumonia que se venía buscando en fuentes externas (NIH ChestX-ray14, etc.), sujeto a confirmar cuánto solapamiento real hay con Chest_x_ray/RSNA vía hash (Sección 6 + notebook de consolidación).
- `Infection Segmentation Data` **no es una fuente aparte** — es un subconjunto con anotación extra de `Lung Segmentation Data`, confirmado por cruce de nombres de archivo, no solo por el texto del dataset.
- Conteo utilizable (Lung Segmentation Data completo): COVID-19 11,956 / Pneumonia 11,263 / Normal 10,701 — un dataset grande y, a diferencia de las fuentes anteriores, ya viene con lung masks para el 100% de las imágenes.
- Sin metadata demográfica de paciente; aporta máscaras de segmentación pulmonar (100%) y de infección (subconjunto) reutilizables para preprocesamiento (recorte de ROI) o explicabilidad.
- Pendiente: correr el cruce de hashes contra Chest_x_ray y RSNA_Pneumonia antes de decidir cuánto de este `Non-COVID`/Pneumonia se suma como "nuevo" al pool combinado del proyecto.